# 07 — Trả lời RQ2: Hồ sơ hành vi người chơi và Phân cụm (C1–C5)

**Mục tiêu:** Xây dựng Behavioral Profile (Design 3), chẩn đoán số cụm K tối ưu (Elbow/Silhouette/DB), thực thi C1-C5 và so sánh outcome sau phân cụm.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


Chọn `runtime` để chạy không cần Drive, hoặc `drive` để 13 notebook dùng chung dữ liệu bền vững. Với `drive`, mọi notebook phải dùng cùng `PUBG_DRIVE_PROJECT_ROOT` và chạy theo thứ tự.

**Chạy nhóm:** chủ thư mục chia sẻ `PUBG_Project` với quyền Editor. Mỗi thành viên thêm shortcut của chính thư mục đó vào My Drive, chọn `drive` và bật `PUBG_REQUIRE_EXISTING_PROJECT = True`. Mỗi người mount Drive của mình; kết quả phải nằm trong cùng thư mục gốc được chia sẻ. Chạy xong notebook, chờ file hiện trên Drive rồi bàn giao cho người tiếp theo; mỗi lần chỉ một người ghi. Người nhận chạy cell cấu hình, Bootstrap và khởi tạo của notebook tiếp theo. Biến trong RAM không được chuyển sang phiên mới; cell đang chạy dở có thể phải chạy lại. Xem `TEAM_DRIVE.md` để thiết lập và xác nhận đường dẫn.


In [ ]:
# @title Chọn nơi lưu dữ liệu { display-mode: "form" }
# @markdown `runtime`: không cần Drive, phù hợp notebook All-in-One.
# @markdown `drive`: lưu nối tiếp 13 notebook trong cùng thư mục Google Drive.
PUBG_STORAGE_MODE = "runtime"  # @param ["runtime", "drive"]
PUBG_DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/PUBG_Project/Project_PUBG"  # @param {type:"string"}
# @markdown Nhóm dùng cùng thư mục đã chia sẻ: bật True để tránh tạo nhầm project riêng khi thiếu shortcut.
PUBG_REQUIRE_EXISTING_PROJECT = False  # @param {type:"boolean"}
# @markdown Số dòng mỗi batch khi đọc CSV trong ZIP; giảm nếu RAM ít. Không lấy mẫu dữ liệu.
PUBG_BATCH_ROWS = 50000  # @param {type:"integer"}


In [ ]:
# Bootstrap: runtime mode needs no Drive; drive mode persists stage outputs.
import base64
import importlib.util
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PUBG_STORAGE_MODE = globals().get("PUBG_STORAGE_MODE", "runtime").strip().lower()
if PUBG_STORAGE_MODE not in {"runtime", "drive"}:
    raise ValueError("PUBG_STORAGE_MODE must be 'runtime' or 'drive'")

if PUBG_STORAGE_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Drive mode is available only on Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(globals().get(
        "PUBG_DRIVE_PROJECT_ROOT", "/content/drive/MyDrive/PUBG_Project/Project_PUBG"
    )).expanduser().resolve()
    if globals().get("PUBG_REQUIRE_EXISTING_PROJECT", False) and not (
        (PROJECT_ROOT / "configs/data.yaml").is_file()
        and (PROJECT_ROOT / "src/utils/config.py").is_file()
    ):
        raise FileNotFoundError(
            "Shared project not found: " + str(PROJECT_ROOT)
            + ". Check Editor access and the PUBG_Project shortcut in My Drive. "
            "No private project was created."
        )
else:
    _candidates = ([Path("/content/Project_PUBG")] if IN_COLAB else
                   [Path.cwd(), *Path.cwd().parents])
    _candidates += [p / "Project_PUBG" for p in list(_candidates)]
    PROJECT_ROOT = next((p.resolve() for p in _candidates
                         if (p / "configs/data.yaml").is_file() and (p / "src/utils/config.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "Project_PUBG"

if not (PROJECT_ROOT / "configs/data.yaml").is_file() or not (PROJECT_ROOT / "src/utils/config.py").is_file():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    _bundle = zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
    for _entry in _bundle.infolist():
        _target = (PROJECT_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Invalid bundled path")
        if not _target.exists():
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_bundle.read(_entry))
    _bundle.close()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if globals().get("PUBG_INSTALL_DEPENDENCIES", IN_COLAB) and not globals().get("_PUBG_PACKAGES_READY", False):
    _requirements = {
        "numpy": "numpy>=1.24.0", "pandas": "pandas>=2.0.0",
        "pyarrow": "pyarrow>=12.0.0", "duckdb": "duckdb>=0.9.0",
        "scipy": "scipy>=1.10.0", "sklearn": "scikit-learn>=1.3.0",
        "yaml": "pyyaml>=6.0",
    }
    _missing = [spec for module, spec in _requirements.items() if importlib.util.find_spec(module) is None]
    if _missing:
        print("Installing missing packages:", ", ".join(_missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", *_missing])
    _PUBG_PACKAGES_READY = True

if PUBG_STORAGE_MODE == "drive":
    os.environ["PUBG_SESSION_DRIVE_ROOT"] = str(PROJECT_ROOT)
    os.environ["PUBG_SESSION_TEMP_DIR"] = str(globals().get("PUBG_RUNTIME_TEMP_DIR", "/content/temp"))
else:
    os.environ.pop("PUBG_SESSION_DRIVE_ROOT", None)
    os.environ.pop("PUBG_SESSION_TEMP_DIR", None)
from src.utils.config import load_config, resolve_paths
cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
for _path in paths.values():
    _path.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Storage:", paths["data_root"], "| Results:", paths["reports_root"])
if PUBG_STORAGE_MODE == "drive":
    print("Storage mode: Google Drive. Stage outputs persist for the next notebook.")
else:
    print("Storage mode: runtime. No Drive authorization required; export before reset.")


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_parquet_df
from src.features.profiles import build_player_behavioral_profiles, filter_profiles_by_retention
from src.analysis.clustering import run_k_diagnostics, execute_rq2_clustering

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)

final_pq = paths["processed"] / "player_match_features.parquet"
df = read_parquet_df(final_pq)

In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
# 1. Xây dựng hồ sơ hành vi người chơi
profiles, outcomes = build_player_behavioral_profiles(df)
filtered_profiles, filtered_outcomes = filter_profiles_by_retention(profiles, outcomes, min_games=5)

In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
# 2. Chẩn đoán K
feature_cols = [c for c in filtered_profiles.columns if c.startswith("mean_") or c.startswith("avg_") or c.endswith("_ratio")]
X = filtered_profiles[feature_cols].values
k_diag = run_k_diagnostics(X, k_range=[2, 3, 4, 5, 6])
print("--- CHẨN ĐOÁN SỐ CỤM K ---")
print(k_diag)

In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
# 3. Phân cụm chính thức C1 và đánh giá C2-C5
selected_k = cfg["rq2"]["n_clusters"] or 4
res = execute_rq2_clustering(filtered_profiles, filtered_outcomes, n_clusters=selected_k, output_dir=paths["reports"] / "tables")
print("--- ĐỐI CHIẾU OUTCOME THEO CỤM (C5) ---")
print(res["outcome_comparison"])